# MiniLM S2 targeted training on OOF-hard human examples

One causal change only: the p85 hardest audit-eligible human train
positives and negatives, mined from 3-fold component/family-disjoint OOF
S2 scores, are deterministically duplicated once. The model,
serialization, optimizer, LR, max length, normalization, and one-epoch
schedule remain fixed. No LLM-labelled data are used.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import pandas as pd
from IPython.display import display

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
TEMP_ROOT = Path('/kaggle/temp/minilm_s2_targeted_hard')
PROJECT_ROOT = TEMP_ROOT / 'product_matching'
PREPARED_DIR = TEMP_ROOT / 'prepared'
MINING_PREP_DIR = TEMP_ROOT / 'mining_prepared'
MODEL_DIR = TEMP_ROOT / 'base_model'
TOKEN_CACHE_ROOT = TEMP_ROOT / 'token_cache'
OOF_RUNS_DIR = TEMP_ROOT / 'oof_runs'
VIEWS_DIR = TEMP_ROOT / 'prepared_views'
BASELINE_TRAINING_DIR = TEMP_ROOT / 'baseline_training'
BASELINE_CHECKPOINT_DIR = TEMP_ROOT / 'baseline_checkpoint'
BASELINE_EVALUATIONS_DIR = WORKING_ROOT / 'minilm_s2_targeted_hard/evaluations/baseline_s2'
HARD_TRAINING_DIR = WORKING_ROOT / 'minilm_s2_targeted_hard/hard_training'
HARD_CHECKPOINT_DIR = WORKING_ROOT / 'minilm_s2_targeted_hard/checkpoint'
HARD_EVALUATIONS_DIR = WORKING_ROOT / 'minilm_s2_targeted_hard/evaluations/targeted_hard_s2'
MINING_OUTPUT_DIR = WORKING_ROOT / 'minilm_s2_targeted_hard/mining'
OUTPUT_DIR = WORKING_ROOT / 'minilm_s2_targeted_hard'
LOGS_DIR = OUTPUT_DIR / 'logs'
CONFIG_PATH = PROJECT_ROOT / 'configs/minilm_s2_targeted_hard_training.json'
EXPECTED_BUNDLE_SHA256 = '198524e896ded0a6f1b020babdeb6ad7e50fac1e296a117fc76c00e24c448861'
EXPECTED_SOURCE_MANIFEST = {'schema_version': 1, 'files': {'configs/minilm_s2_targeted_hard_training.json': {'bytes': 1292, 'sha256': 'f33e52e0443bfd2d6cece0c9e3274045b2a7563f1f13066bd7131eb3be6a17a6'}, 'requirements-serialization-ablation.txt': {'bytes': 166, 'sha256': 'ad14f13456ec89586d78f3174efe8937d5798b58aa7d608f6cc47f232806b16a'}, 'requirements-hard-mining.txt': {'bytes': 77, 'sha256': '28e1c4b043fbd51e33a03eeac4ad8972722d10a02f7bc7c007ca101411461dbb'}, 'src/__init__.py': {'bytes': 62, 'sha256': '1ebfb0504084e6c7b27bb3a60370eb9f01bdf6ac9a801bc400a58da4d8d674eb'}, 'src/cheap_ensemble.py': {'bytes': 13915, 'sha256': 'b9fee08e2a07d2424fd33f39c87bc250d73d1966ee6354a16e1ad4848f253afc'}, 'src/cross_encoder_training.py': {'bytes': 12254, 'sha256': '97d9948b753a18a76975c043a216ca0d26f56eeffd61d5d4027becd98159b619'}, 'src/qwen_reranker.py': {'bytes': 3922, 'sha256': '079396d0de4f64564f21e61d7a58be028a0439570fab3bbc2fc08837e1229804'}, 'src/qwen_training.py': {'bytes': 14754, 'sha256': '45bcd4da9de515ef93dde144781ad89a01515d3e82be37b17848d65c41c330af'}, 'src/serialization_ablation.py': {'bytes': 17772, 'sha256': '98323abb037d37dd9baaf5eb7fd83e0bf0e12f096256290cecbbd73a60085e3a'}, 'scripts/train_serialization_ablation.py': {'bytes': 16774, 'sha256': '7c7b7f7098561ee3acc7e4f159a7046aebf460ab4a703459f34a88b6d039548f'}, 'scripts/prepare_minilm_s0_s2_new_splits.py': {'bytes': 5601, 'sha256': 'af1ccf2f7ac280bd97710e504a50722b272d27a9cc4fc39316506c6b562c8545'}, 'scripts/prepare_minilm_s2_hard_mining.py': {'bytes': 11354, 'sha256': 'ca038db7f5ab9dc57f27b89e9e29e461027e9141432519920ba7ef004bfd3e95'}, 'scripts/evaluate_minilm_new_splits.py': {'bytes': 5601, 'sha256': '478b0968679e3f1a7a1eef33d3278ef35133f7e4daeea3671f6a9d2dda92deb1'}, 'scripts/mine_minilm_s2_oof_hard_examples.py': {'bytes': 9021, 'sha256': 'df1308aef67267a3a50ab0c87770b4f9f0fd37dd118576d8432a858480dd1839'}, 'scripts/summarize_minilm_s2_hard_training.py': {'bytes': 8131, 'sha256': '8eb53cc6ad0aa7401292fe53257c9d53a6356e7535e0a7e1a9fba0d58724f067'}}}
EXPECTED_ASSIGNMENTS_SHA256 = '39c1a184b9795b64237bb61a8aadccdf1d4daa91fbac55a7626374f76a93b554'
EXPECTED_SLICE_FLAGS_SHA256 = 'e255ca7ab9cfbfea15e70d4d011ac5708f8eb46046e6cb0cd4cd1e878ecd2ee8'
RAW_DATASET_REF = 'dinakepecheva/e-cup-human-data'
VALIDATION_DATASET_REF = 'alexproger23/product-matching-validation-splits-v1'
CODE_DATASET_REF = 'dinakepecheva/product-matching-minilm-s2-targeted-hard-code'

def exactly_one(filename):
    candidates = list(INPUT_ROOT.glob(f'**/{filename}'))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one {filename!r}, found {candidates}')
    return candidates[0]

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

items_path = exactly_one('items_human.parquet')
train_path = exactly_one('human_train_pairs.parquet')
iid_path = exactly_one('human_iid_validation_pairs.parquet')
hard_path = exactly_one('human_hard_validation_pairs.parquet')
ood_path = exactly_one('human_ood_validation_pairs.parquet')
assignments_path = exactly_one('hard_audit_assignments.csv')
if sha256(assignments_path) != EXPECTED_ASSIGNMENTS_SHA256:
    raise RuntimeError('Hard audit assignments SHA-256 mismatch')
hard_clean_slice_flags_path = exactly_one('hard_clean_slice_flags.csv')
if sha256(hard_clean_slice_flags_path) != EXPECTED_SLICE_FLAGS_SHA256:
    raise RuntimeError('Hard-clean slice flags SHA-256 mismatch')
bundle_candidates = list(INPUT_ROOT.glob('**/minilm_s2_targeted_hard_code.zip'))
bundle_candidates.extend(
    path for path in INPUT_ROOT.glob('**/minilm_s2_targeted_hard_code') if path.is_dir()
)
if len(bundle_candidates) != 1:
    raise RuntimeError(f'Expected one code bundle, found {bundle_candidates}')
bundle_path = bundle_candidates[0]
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
if bundle_path.is_file():
    if sha256(bundle_path) != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError('Code bundle SHA-256 mismatch')
    with zipfile.ZipFile(bundle_path) as archive:
        for member in archive.namelist():
            relative = PurePosixPath(member)
            if relative.is_absolute() or '..' in relative.parts:
                raise RuntimeError(f'Unsafe bundle member: {member}')
        archive.extractall(PROJECT_ROOT)
else:
    shutil.copytree(bundle_path, PROJECT_ROOT, dirs_exist_ok=True)
source_manifest = json.loads((PROJECT_ROOT / 'source_manifest.json').read_text(encoding='utf-8'))
if source_manifest != EXPECTED_SOURCE_MANIFEST:
    raise RuntimeError('Expanded source manifest mismatch')
for relative, expected in source_manifest['files'].items():
    source = PROJECT_ROOT.joinpath(*PurePosixPath(relative).parts)
    if sha256(source) != expected['sha256']:
        raise RuntimeError(f'Source hash mismatch: {relative}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)
print('items:', items_path)
print('train:', train_path)
print('code dataset:', CODE_DATASET_REF)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
from datetime import datetime, timezone
import uuid

RUN_ID_PATH = WORKING_ROOT / "experiment_run_id.txt"
RUN_STARTED_PATH = WORKING_ROOT / "experiment_started_at_utc.txt"
if RUN_ID_PATH.is_file():
    EXPERIMENT_RUN_ID = RUN_ID_PATH.read_text(encoding="utf-8").strip()
else:
    EXPERIMENT_RUN_ID = uuid.uuid4().hex
    RUN_ID_PATH.write_text(EXPERIMENT_RUN_ID + "\n", encoding="utf-8")
if RUN_STARTED_PATH.is_file():
    EXPERIMENT_STARTED_AT_UTC = RUN_STARTED_PATH.read_text(
        encoding="utf-8"
    ).strip()
else:
    EXPERIMENT_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat(
        timespec="seconds"
    ).replace("+00:00", "Z")
    RUN_STARTED_PATH.write_text(
        EXPERIMENT_STARTED_AT_UTC + "\n", encoding="utf-8"
    )
print(
    json.dumps(
        {
            "run_id": EXPERIMENT_RUN_ID,
            "started_at_utc": EXPERIMENT_STARTED_AT_UTC,
        },
        ensure_ascii=False,
    )
)

## Frozen configuration and base model

In [ ]:
EXPECTED_CONFIG = {'experiment': 'minilm_s2_targeted_hard_training', 'model': 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', 'variant': 'S2_VALUES_ONLY', 'variants': ['S2_VALUES_ONLY'], 'validation_splits': {'iid': 'iid_pairs.parquet', 'hard_clean': 'hard_clean_pairs.parquet', 'ood': 'ood_pairs.parquet'}, 'expected_pairs': {'train': 306669, 'iid': 12000, 'hard': 5814, 'hard_all': 5814, 'hard_clean': 4929, 'ood': 41171}, 'oof_folds': 3, 'oof_group_seed': 20260816, 'hard_quantile': 0.85, 'hard_oversample_factor': 2, 'maximum_iid_macro_ap_drop': 0.01, 'epochs': 1, 'batch_size': 64, 'eval_batch_size': 192, 'gradient_accumulation': 1, 'learning_rate': 2e-05, 'weight_decay': 0.01, 'warmup_ratio': 0.05, 'max_length': 256, 'attention_implementation': 'sdpa', 'bucket_size_multiplier': 50, 'dataloader_workers': 2, 'prefetch_factor': 2, 'tokenization_batch_size': 512, 'tokenization_log_every': 50, 'gradient_checkpointing': False, 'symmetric_validation': True, 'label_smoothing': 0.0, 'max_grad_norm': 1.0, 'log_every': 50, 'hybrid_frequency': {'strategy': 'occurrence_coverage', 'target_coverage': 0.8, 'minimum_item_support': 100, 'maximum_frequent_keys': 1024}, 'seed': 42}
TRAIN_CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
if TRAIN_CONFIG != EXPECTED_CONFIG:
    raise RuntimeError('Notebook config and bundled config differ')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
     '--upgrade-strategy', 'only-if-needed', '-r',
     str(PROJECT_ROOT / 'requirements-hard-mining.txt')],
    check=True,
)
from huggingface_hub import HfApi, snapshot_download
model_revision = HfApi().model_info(TRAIN_CONFIG['model']).sha
snapshot_download(
    repo_id=TRAIN_CONFIG['model'], revision=model_revision, local_dir=MODEL_DIR
)
print('model revision:', model_revision)

## Prepare fixed S2 texts, clean-hard, train audit, and 3 OOF folds

In [ ]:
preparation_started = time.perf_counter()
subprocess.run(
    [sys.executable, '-u', str(PROJECT_ROOT / 'scripts/prepare_minilm_s0_s2_new_splits.py'),
     '--items', str(items_path), '--train', str(train_path), '--iid', str(iid_path),
     '--hard', str(hard_path), '--ood', str(ood_path), '--config', str(CONFIG_PATH),
     '--output-dir', str(PREPARED_DIR)],
    check=True, cwd=PROJECT_ROOT,
)
subprocess.run(
    [sys.executable, '-u', str(PROJECT_ROOT / 'scripts/prepare_minilm_s2_hard_mining.py'),
     '--config', str(CONFIG_PATH), '--items', str(items_path),
     '--train-pairs', str(train_path), '--hard-audit-assignments', str(assignments_path),
     '--output-dir', str(MINING_PREP_DIR)],
    check=True, cwd=PROJECT_ROOT,
)
shutil.copy2(MINING_PREP_DIR / 'hard_clean_pairs.parquet', PREPARED_DIR / 'hard_clean_pairs.parquet')
preparation_seconds = time.perf_counter() - preparation_started
prep_report = json.loads((MINING_PREP_DIR / 'preparation_report.json').read_text(encoding='utf-8'))
display(pd.DataFrame(prep_report['oof_folds']))
print('preparation seconds:', preparation_seconds)

## Helpers for isolated single-GPU jobs

In [ ]:
def make_view(name, train_pairs, validation_pairs):
    directory = VIEWS_DIR / name
    directory.mkdir(parents=True, exist_ok=True)
    links = {
        'items.parquet': PREPARED_DIR / 'items.parquet',
        'train_pairs.parquet': Path(train_pairs),
        'validation_pairs.parquet': Path(validation_pairs),
    }
    for filename, source in links.items():
        target = directory / filename
        target.unlink(missing_ok=True)
        target.symlink_to(source.resolve())
    return directory

def launch_training(name, gpu, prepared_view, output_dir, checkpoint_dir):
    log_path = LOGS_DIR / f'{name}.log'
    handle = log_path.open('w', encoding='utf-8', buffering=1)
    command = [
        sys.executable, '-u', str(PROJECT_ROOT / 'scripts/train_serialization_ablation.py'),
        '--config', str(CONFIG_PATH), '--prepared-dir', str(prepared_view),
        '--model-path', str(MODEL_DIR), '--model-revision', model_revision,
        '--variant', TRAIN_CONFIG['variant'], '--output-dir', str(output_dir),
        '--checkpoint-dir', str(checkpoint_dir),
        '--token-cache-dir', str(TOKEN_CACHE_ROOT / name),
    ]
    environment = os.environ.copy()
    environment['CUDA_VISIBLE_DEVICES'] = str(gpu)
    process = subprocess.Popen(
        command, cwd=PROJECT_ROOT, env=environment,
        stdout=handle, stderr=subprocess.STDOUT, text=True,
    )
    print(f'launched {name} on GPU {gpu}: pid={process.pid}', flush=True)
    return name, process, handle, log_path

def launch_evaluation(name, gpu, checkpoint_dir, training_report, output_dir):
    log_path = LOGS_DIR / f'{name}.log'
    handle = log_path.open('w', encoding='utf-8', buffering=1)
    command = [
        sys.executable, '-u', str(PROJECT_ROOT / 'scripts/evaluate_minilm_new_splits.py'),
        '--config', str(CONFIG_PATH), '--prepared-dir', str(PREPARED_DIR),
        '--checkpoint-dir', str(checkpoint_dir), '--training-report', str(training_report),
        '--variant', TRAIN_CONFIG['variant'], '--output-dir', str(output_dir),
        '--token-cache-dir', str(TOKEN_CACHE_ROOT / name),
    ]
    environment = os.environ.copy()
    environment['CUDA_VISIBLE_DEVICES'] = str(gpu)
    process = subprocess.Popen(
        command, cwd=PROJECT_ROOT, env=environment,
        stdout=handle, stderr=subprocess.STDOUT, text=True,
    )
    print(f'launched {name} on GPU {gpu}: pid={process.pid}', flush=True)
    return name, process, handle, log_path

def wait_jobs(jobs, poll_seconds=60):
    while any(process.poll() is None for _, process, _, _ in jobs):
        time.sleep(poll_seconds)
        print('job status:', {name: process.poll() for name, process, _, _ in jobs}, flush=True)
    failures = []
    for name, process, handle, log_path in jobs:
        handle.close()
        if process.returncode:
            failures.append({
                'name': name, 'returncode': process.returncode,
                'tail': log_path.read_text(encoding='utf-8', errors='replace')[-16000:],
            })
    if failures:
        raise RuntimeError('GPU job failure:\n' + json.dumps(failures, ensure_ascii=False, indent=2))

## OOF wave 1: folds 0 and 1

In [ ]:
experiment_started = time.perf_counter()
fold_jobs = []
for fold, gpu in ((0, 0), (1, 1)):
    view = make_view(
        f'oof_fold_{fold}',
        MINING_PREP_DIR / f'oof_fold_{fold}_train_pairs.parquet',
        MINING_PREP_DIR / f'oof_fold_{fold}_validation_pairs.parquet',
    )
    fold_jobs.append(
        launch_training(
            f'oof_fold_{fold}', gpu, view,
            OOF_RUNS_DIR / f'fold_{fold}', TEMP_ROOT / f'oof_checkpoint_{fold}'
        )
    )
wait_jobs(fold_jobs)
for fold in (0, 1):
    shutil.rmtree(TEMP_ROOT / f'oof_checkpoint_{fold}', ignore_errors=True)

## OOF wave 2: fold 2 and causal baseline A in parallel

In [ ]:
fold2_view = make_view(
    'oof_fold_2', MINING_PREP_DIR / 'oof_fold_2_train_pairs.parquet',
    MINING_PREP_DIR / 'oof_fold_2_validation_pairs.parquet'
)
baseline_view = make_view(
    'baseline', PREPARED_DIR / 'train_pairs.parquet', PREPARED_DIR / 'iid_pairs.parquet'
)
wave2 = [
    launch_training(
        'oof_fold_2', 0, fold2_view, OOF_RUNS_DIR / 'fold_2',
        TEMP_ROOT / 'oof_checkpoint_2'
    ),
    launch_training(
        'baseline_s2', 1, baseline_view, BASELINE_TRAINING_DIR,
        BASELINE_CHECKPOINT_DIR
    ),
]
wait_jobs(wave2)
shutil.rmtree(TEMP_ROOT / 'oof_checkpoint_2', ignore_errors=True)

## OOF hardness mining and deterministic x2 oversampling

In [ ]:
subprocess.run(
    [sys.executable, '-u', str(PROJECT_ROOT / 'scripts/mine_minilm_s2_oof_hard_examples.py'),
     '--config', str(CONFIG_PATH),
     '--train-audit', str(MINING_PREP_DIR / 'train_label_audit_and_folds.parquet'),
     '--oof-runs-dir', str(OOF_RUNS_DIR), '--output-dir', str(MINING_OUTPUT_DIR)],
    check=True, cwd=PROJECT_ROOT,
)
mining_report = json.loads((MINING_OUTPUT_DIR / 'hard_mining_report.json').read_text(encoding='utf-8'))
display(pd.DataFrame(mining_report['counts']))
display(pd.read_csv(MINING_OUTPUT_DIR / 'hardness_distribution_quantiles.csv'))

## Train B and evaluate A in parallel

In [ ]:
hard_view = make_view(
    'targeted_hard', MINING_OUTPUT_DIR / 'train_pairs_hard_x2.parquet',
    PREPARED_DIR / 'iid_pairs.parquet'
)
wave3 = [
    launch_training(
        'targeted_hard_s2', 0, hard_view, HARD_TRAINING_DIR, HARD_CHECKPOINT_DIR
    ),
    launch_evaluation(
        'baseline_evaluation', 1, BASELINE_CHECKPOINT_DIR,
        BASELINE_TRAINING_DIR / 'training_report.json', BASELINE_EVALUATIONS_DIR
    ),
]
wait_jobs(wave3)

## Evaluate B and summarize the causal comparison

In [ ]:
hard_eval = launch_evaluation(
    'targeted_hard_evaluation', 0, HARD_CHECKPOINT_DIR,
    HARD_TRAINING_DIR / 'training_report.json', HARD_EVALUATIONS_DIR
)
wait_jobs([hard_eval])
subprocess.run(
    [sys.executable, '-u', str(PROJECT_ROOT / 'scripts/summarize_minilm_s2_hard_training.py'),
     '--config', str(CONFIG_PATH),
     '--baseline-evaluations', str(BASELINE_EVALUATIONS_DIR),
     '--hard-evaluations', str(HARD_EVALUATIONS_DIR),
     '--hard-clean-audit', str(hard_clean_slice_flags_path),
     '--mining-report', str(MINING_OUTPUT_DIR / 'hard_mining_report.json'),
     '--output-dir', str(OUTPUT_DIR)],
    check=True, cwd=PROJECT_ROOT,
)

## Results

In [ ]:
aggregate = json.loads((OUTPUT_DIR / 'training_report.json').read_text(encoding='utf-8'))
display(pd.read_csv(OUTPUT_DIR / 'main_metrics.csv'))
display(pd.read_csv(OUTPUT_DIR / 'metric_deltas.csv'))
display(pd.read_csv(OUTPUT_DIR / 'hard_clean_slice_comparison.csv'))
experiment_wall_seconds = time.perf_counter() - experiment_started
completed_at = datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z')
sheets_report = dict(aggregate['reports']['targeted_hard_s2']['hard_clean'])
sheets_report['full_experiment_report'] = aggregate
completion = {
    'status': 'complete', 'run_id': EXPERIMENT_RUN_ID,
    'started_at_utc': EXPERIMENT_STARTED_AT_UTC,
    'completed_at_utc': completed_at,
    'experiment': TRAIN_CONFIG['experiment'], 'model': TRAIN_CONFIG['model'],
    'dataset_ref': VALIDATION_DATASET_REF,
    'kaggle_kernel_ref': os.getenv('KAGGLE_KERNEL_RUN_ID') or os.getenv('KAGGLE_KERNEL_INFERENCE_RUN_ID') or '',
    'code_bundle_sha256': EXPECTED_BUNDLE_SHA256,
    'training_wall_seconds': experiment_wall_seconds,
    'configuration': TRAIN_CONFIG,
    'training_report': sheets_report,
}
(OUTPUT_DIR / 'run_completion.json').write_text(
    json.dumps(completion, ensure_ascii=False, indent=2), encoding='utf-8'
)

## Completion marker

In [ ]:
notebook_completed = {
    **completion,
    'success_gate': aggregate['success_gate'],
    'deltas_macro_average_precision': aggregate['deltas_macro_average_precision'],
    'google_sheets_status': 'disabled_by_user',
}
(WORKING_ROOT / 'notebook_completed.json').write_text(
    json.dumps(notebook_completed, ensure_ascii=False, indent=2), encoding='utf-8'
)
(OUTPUT_DIR / 'COMPLETED').write_text('complete\n', encoding='utf-8')
shutil.rmtree(BASELINE_CHECKPOINT_DIR, ignore_errors=True)
print(json.dumps(notebook_completed, ensure_ascii=False, indent=2))